# SCF convergence threshold and force accuracy

The SCF loop in `pw.x` stops when the change in total energy between two successive
iterations falls below `conv_thr` (Ry).  Because the Hellmann–Feynman forces require
exact Kohn–Sham wavefunctions, any residual error in the electron density propagates
into the computed forces.

`pw.x` estimates this error and prints it alongside the forces:

```
Total force = X.XXXXXX     Total SCF correction = Y.YYYYYY
```

The *total* SCF correction is a vector sum over all atoms and can vanish by symmetry.
With `verbosity = 'medium'`, `pw.x` also prints the correction per atom, allowing a
mean-absolute-error metric that does not cancel.

### Exercises

1. How does the mean absolute SCF force correction scale with `conv_thr`?  Is the
   relationship what you expect from perturbation theory?
2. Compare the SCF correction (QE's own estimate) with the actual force change
   `|ΔFz|` relative to the tightest threshold.  Is the estimate reliable?
3. At what `conv_thr` is the force error below your target accuracy (10 meV/Å)?
   What is the computational cost compared to the default `conv_thr = 1e-6`?

In [ ]:
from pathlib import Path
import glob, shutil

RUN_ROOT = Path('.').resolve()

_pw_candidates = sorted(glob.glob('/home/pietro/repositories/q-e/build_test_gcc/bin/pw.x'))
PW_CMD = [_pw_candidates[0]] if _pw_candidates else (
    [shutil.which('pw.x')] if shutil.which('pw.x') else None
)
if PW_CMD is None:
    raise RuntimeError('pw.x not found. Check QE installation or PATH.')

PSEUDO_DIR = RUN_ROOT / 'pseudo'
OUT_DIR    = RUN_ROOT / 'out'
CONV_DIR   = RUN_ROOT / 'convergence' / 'conv_thr'

for d in [OUT_DIR, CONV_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('pw.x:', PW_CMD[0])

In [ ]:
from ase.build import bulk

from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    AtomicSpeciesCard, AtomicPositionsCard, KPointsAutoCard, PWInput,
)
from convergence_runner import (
    QERunner, RY_TO_EV,
    first_globally_converged_index,
)
from convergence_plotting import plot_conv_thr_sweep

PSEUDOS = {'Mg': 'Mg.upf', 'O': 'O.upf'}

missing = [f for f in PSEUDOS.values() if not (PSEUDO_DIR / f).is_file()]
if missing:
    raise FileNotFoundError(f'Missing pseudopotentials: {missing}')

atoms = bulk('MgO', 'rocksalt', a=4.21)
nat   = len(atoms)
print('System: MgO rocksalt, nat =', nat)

In [ ]:
def build_mgo_input(atoms, ecutwfc, nk, prefix, conv_thr):
    control   = ControlNamelist(
        calculation='scf', prefix=prefix,
        pseudo_dir=str(PSEUDO_DIR), outdir=str(OUT_DIR),
        tprnfor=True, verbosity='medium',
    )
    system    = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ecutwfc)
    electrons = ElectronsNamelist(conv_thr=conv_thr)
    species   = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS)
    positions = AtomicPositionsCard.from_atoms(atoms, units='crystal')
    kpoints   = KPointsAutoCard(2, nk=nk)
    return PWInput(
        control=control, system=system, electrons=electrons,
        atomic_species=species, atomic_positions=positions, k_points=kpoints,
    )

In [ ]:
# Student input — use the ecutwfc and nk you found to be converged in the
# previous notebook.
ECUTWFC   = 80
NK        = 4
FORCE_THRESHOLD_MEV_PER_ANG = 10.0

DISPLACED_ATOM_INDEX_1BASED = 1
DISPLACEMENT_ANG            = 0.01   # Å — small enough that SCF error is visible

CONV_THR_VALUES = [1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]

# Set True to rerun pw.x even when an output file already exists.
FORCE_RERUN = True

print(f'ecutwfc = {ECUTWFC} Ry,  k-mesh = {NK}x{NK}x{NK}')
print(f'Displacement: {DISPLACEMENT_ANG:.4f} Ang along z')

In [ ]:
atoms_displaced = atoms.copy()
pos = atoms_displaced.get_positions()
pos[DISPLACED_ATOM_INDEX_1BASED - 1, 2] += DISPLACEMENT_ANG
atoms_displaced.set_positions(pos)

runner = QERunner(PW_CMD)

cases = [
    (f'conv_thr_{i}', build_mgo_input(
        atoms_displaced, ecutwfc=ECUTWFC, nk=NK,
        prefix='mgo_conv_thr', conv_thr=thr,
    ))
    for i, thr in enumerate(CONV_THR_VALUES)
]

results = runner.run_sweep(
    cases, CONV_DIR,
    force_rerun=FORCE_RERUN,
)

In [ ]:
fz          = [r['forces_ev_ang'][DISPLACED_ATOM_INDEX_1BASED - 1, 2]      for r in results]
scf_corr    = [(np.mean(np.abs(r['scf_corrections_ev_ang'])) if r['scf_corrections_ev_ang'] is not None else 0.0) for r in results]
time_s      = [r['wall_s']              for r in results]
dF_mev      = [abs(f - fz[-1]) * 1000  for f in fz]

idx = first_globally_converged_index(dF_mev, FORCE_THRESHOLD_MEV_PER_ANG)
conv_thr_conv = CONV_THR_VALUES[idx] if idx is not None else CONV_THR_VALUES[-1]

plot_conv_thr_sweep(
    CONV_THR_VALUES, fz, scf_corr, dF_mev, time_s,
    force_threshold_mev_ang=FORCE_THRESHOLD_MEV_PER_ANG,
)
print(f'Force converged to {FORCE_THRESHOLD_MEV_PER_ANG} meV/Å at conv_thr = {conv_thr_conv:.0e} Ry')

---

## Silicon (diamond)

Same exercise repeated for Si in the diamond structure.  Si has a norm-conserving
pseudo, so the setup is analogous to MgO.  The primitive cell (ibrav = 2, FCC)
contains two Si atoms; we displace atom 1 along z.

In [ ]:
def build_si_input(atoms, ecutwfc, nk, prefix, conv_thr):
    control   = ControlNamelist(
        calculation='scf', prefix=prefix,
        pseudo_dir=str(PSEUDO_DIR), outdir=str(OUT_DIR),
        tprnfor=True, verbosity='medium',
    )
    system    = SystemNamelist.from_atoms(atoms, ibrav=2, ecutwfc=ecutwfc)
    electrons = ElectronsNamelist(conv_thr=conv_thr)
    species   = AtomicSpeciesCard.from_atoms(atoms, PSEUDOS_SI)
    positions = AtomicPositionsCard.from_atoms(atoms, units='crystal')
    kpoints   = KPointsAutoCard(2, nk=nk)
    return PWInput(
        control=control, system=system, electrons=electrons,
        atomic_species=species, atomic_positions=positions, k_points=kpoints,
    )

In [ ]:
PSEUDOS_SI  = {'Si': 'Si.upf'}
atoms_si    = bulk('Si', 'diamond', a=5.43)

missing_si = [f for f in PSEUDOS_SI.values() if not (PSEUDO_DIR / f).is_file()]
if missing_si:
    raise FileNotFoundError(f'Missing pseudopotentials: {missing_si}')

ECUTWFC_SI   = 40
NK_SI        = 4
DISPLACEMENT_ANG_SI          = 0.01
DISPLACED_ATOM_INDEX_SI      = 1
FORCE_THRESHOLD_MEV_PER_ANG_SI = 10.0

CONV_THR_VALUES_SI = [1e-6, 1e-7, 1e-8, 1e-9, 1e-10, 1e-11, 1e-12, 1e-13]

CONV_DIR_SI  = RUN_ROOT / 'convergence' / 'conv_thr_si'
CONV_DIR_SI.mkdir(parents=True, exist_ok=True)

FORCE_RERUN_SI = True

print(f'System: Si diamond, nat = {len(atoms_si)}')
print(f'ecutwfc = {ECUTWFC_SI} Ry,  k-mesh = {NK_SI}x{NK_SI}x{NK_SI}')
print(f'Displacement: {DISPLACEMENT_ANG_SI:.4f} Ang along z')

In [ ]:
atoms_si_displaced = atoms_si.copy()
pos_si = atoms_si_displaced.get_positions()
pos_si[DISPLACED_ATOM_INDEX_SI - 1, 2] += DISPLACEMENT_ANG_SI
atoms_si_displaced.set_positions(pos_si)

cases_si = [
    (f'conv_thr_{i}', build_si_input(
        atoms_si_displaced, ecutwfc=ECUTWFC_SI, nk=NK_SI,
        prefix='si_conv_thr', conv_thr=thr,
    ))
    for i, thr in enumerate(CONV_THR_VALUES_SI)
]

results_si = runner.run_sweep(
    cases_si, CONV_DIR_SI,
    force_rerun=FORCE_RERUN_SI,
)

In [ ]:
fz_si       = [r['forces_ev_ang'][DISPLACED_ATOM_INDEX_SI - 1, 2]        for r in results_si]
scf_corr_si = [(np.mean(np.abs(r['scf_corrections_ev_ang'])) if r['scf_corrections_ev_ang'] is not None else 0.0) for r in results_si]
time_si     = [r['wall_s']                for r in results_si]
dF_mev_si   = [abs(f - fz_si[-1]) * 1000 for f in fz_si]

idx_si = first_globally_converged_index(dF_mev_si, FORCE_THRESHOLD_MEV_PER_ANG_SI)
conv_thr_conv_si = CONV_THR_VALUES_SI[idx_si] if idx_si is not None else CONV_THR_VALUES_SI[-1]

plot_conv_thr_sweep(
    CONV_THR_VALUES_SI, fz_si, scf_corr_si, dF_mev_si, time_si,
    force_threshold_mev_ang=FORCE_THRESHOLD_MEV_PER_ANG_SI,
)
print(f'Si: force converged to {FORCE_THRESHOLD_MEV_PER_ANG_SI} meV/Å at conv_thr = {conv_thr_conv_si:.0e} Ry')